In [1]:
import pandas as pd
import numpy as np

**Step 1: Reshape the Data so that it appears in the following format**
- Zip Code
- Month
- Pounds Distributed 
- Source 

In [2]:
xls = pd.read_excel("FOOD_BANKS_DATA.xlsx", sheet_name=None)
fsd = xls["FSD_FOOD"]
sdfb = xls["SDFB_FOOD"]

In [3]:
def combine_food(df, source_name):
    df = df.copy()
    #drop the total column, no need for a row sum column in long format
    total_col = [c for c in df.columns if 'Total' in str(c)][0]
    df = df.drop(columns=[total_col])
    
    # every column excep tdate becomes a zip code row
    long_df = df.melt(id_vars="Date",
                     var_name="Zip Code",
                     value_name="Pounds_Distributed")

    #stnadardize date to mmyy
    long_df["Month Year"] = pd.to_datetime(long_df["Date"]).dt.strftime("%Y-%m")
    long_df = long_df.drop(columns=["Date"])
    
    #tag the source
    long_df["Source"] = source_name

    #reorder columns to match the given schema
    long_df = long_df[["Zip Code", "Month Year", "Pounds_Distributed", "Source"]]
    
    return long_df

fsd_long = combine_food(fsd, "FSD")
sdfb_long = combine_food(sdfb, "SDFB")

food_banks_long = pd.concat([fsd_long, sdfb_long], ignore_index=True)

In [4]:
food_banks_long

,Zip Code,Month Year,Pounds_Distributed,Source
0,91901,2018-01,7884.0,FSD
1,91901,2018-02,10558.0,FSD
2,91901,2018-03,10642.0,FSD
3,91901,2018-04,12192.0,FSD
4,91901,2018-05,10370.0,FSD
...,...,...,...,...
17199,92007,2025-04,0.0,SDFB
17200,92007,2025-05,0.0,SDFB
17201,92007,2025-06,0.0,SDFB
17202,92007,2025-07,0.0,SDFB


In [5]:
food_banks_long.dtypes

Zip Code                  str
Month Year                str
Pounds_Distributed    float64
Source                    str
dtype: object

**Standardize the ZIp Codes**

In [6]:
def clean_zip(z):
    if pd.isna(z): return pd.NA
    z_str = str(z).strip()
    if z_str in ("", "nan", "00nan"): return pd.NA
    try: return str(int(float(z_str))).zfill(5)
    except ValueError: return pd.NA

food_banks_long["Zip Code"] = food_banks_long["Zip Code"].apply(clean_zip)

**Check for Missing Values or Duplicates**

In [7]:
#Check for Nulls

In [8]:
food_banks_long.isna().sum()

Zip Code              0
Month Year            0
Pounds_Distributed    0
Source                0
dtype: int64

In [9]:
#check for duplicates

In [10]:
dupe_check = food_banks_long.groupby(["Zip Code", "Month Year", "Source"]).size()
print(f"{(dupe_check > 1).sum()} zip-month-source combos have more than one row")

0 zip-month-source combos have more than one row


**Filter for Zip Codes Within San Diego Region**

In [11]:
san_diego_zips_standardized = [
    "91901", "91902", "91903", "91905", "91906", "91908", "91909", "91910",
    "91911", "91912", "91913", "91914", "91915", "91916", "91917", "91921",
    "91931", "91932", "91933", "91934", "91935", "91941", "91942", "91943",
    "91944", "91945", "91946", "91950", "91951", "91962", "91963", "91976",
    "91977", "91978", "91979", "91980", "92003", "92004", "92007", "92008",
    "92009", "92010", "92011", "92013", "92014", "92018", "92019", "92020",
    "92021", "92022", "92023", "92024", "92025", "92026", "92027", "92028",
    "92029", "92030", "92033", "92036", "92037", "92038", "92039", "92040",
    "92046", "92049", "92051", "92052", "92054", "92055", "92056", "92057",
    "92058", "92059", "92060", "92061", "92064", "92065", "92066", "92067",
    "92068", "92069", "92070", "92071", "92072", "92074", "92075", "92078",
    "92079", "92081", "92082", "92083", "92084", "92085", "92086", "92088",
    "92091", "92092", "92093", "92101", "92102", "92103", "92104", "92105",
    "92106", "92107", "92108", "92109", "92110", "92111", "92112", "92113",
    "92114", "92115", "92116", "92117", "92118", "92119", "92120", "92121",
    "92122", "92123", "92124", "92126", "92127", "92128", "92129", "92130",
    "92131", "92136", "92137", "92138", "92139", "92140", "92142", "92143",
    "92145", "92149", "92150", "92153", "92154", "92159", "92160", "92163",
    "92165", "92166", "92167", "92168", "92169", "92170", "92171", "92172",
    "92173", "92174", "92175", "92176", "92177", "92178", "92186", "92191",
    "92192", "92193", "92195", "92196", "92198", 
]

In [12]:
sd_zip_strs = set(str(z).zfill(5) for z in san_diego_zips_standardized)

In [13]:
food_banks_long["Zip_Code_In_San_Diego_County"] = food_banks_long["Zip Code"].isin(sd_zip_strs)

In [14]:
food_banks_long_sd_records = (
    food_banks_long[food_banks_long["Zip_Code_In_San_Diego_County"]]
    .drop(columns="Zip_Code_In_San_Diego_County")
    .copy()
)

In [15]:
invalid_records = (
    food_banks_long[~food_banks_long["Zip_Code_In_San_Diego_County"]]
    .drop(columns="Zip_Code_In_San_Diego_County")
    .copy()
)

In [16]:
food_banks_long_sd_records

,Zip Code,Month Year,Pounds_Distributed,Source
0,91901,2018-01,7884.0,FSD
1,91901,2018-02,10558.0,FSD
2,91901,2018-03,10642.0,FSD
3,91901,2018-04,12192.0,FSD
4,91901,2018-05,10370.0,FSD
...,...,...,...,...
17199,92007,2025-04,0.0,SDFB
17200,92007,2025-05,0.0,SDFB
17201,92007,2025-06,0.0,SDFB
17202,92007,2025-07,0.0,SDFB


**Check for Outliers and Negative Values**
- There are two values with negative pounds distributed, need to follow up and ask more questions

In [17]:
negative = food_banks_long_sd_records[
    (food_banks_long_sd_records["Pounds_Distributed"] < 0)
]

In [18]:
negative

,Zip Code,Month Year,Pounds_Distributed,Source
5898,92084,2018-11,-29863.0,FSD
6849,92108,2021-06,-1857.0,FSD


In [19]:
def flag_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    is_outlier = (df[column] < lower_bound) | (df[column] > upper_bound)
    return is_outlier, lower_bound, upper_bound

outlier_cols = []
for col in ["Pounds_Distributed"]:
    outlier_flag, lower, upper = flag_outliers(food_banks_long_sd_records, col)
    outlier_col_name = f"{col}_Outlier"
    food_banks_long_sd_records[outlier_col_name] = outlier_flag
    outlier_cols.append(outlier_col_name)
    n_outliers = outlier_flag.sum()
    print(f"{col}: bounds [{lower:.2f}, {upper:.2f}] — "
          f"{n_outliers} outliers ({n_outliers/len(food_banks_long_sd_records)*100:.1f}%)")

food_banks_long_sd_records["Any_Outlier"] = food_banks_long_sd_records[outlier_cols].any(axis=1)

print(f"\n{food_banks_long_sd_records['Any_Outlier'].sum()} of {len(food_banks_long_sd_records)} rows "
      f"flagged as an outlier on at least one column")

Pounds_Distributed: bounds [-59583.22, 102461.81] — 1406 outliers (8.2%)

1406 of 17204 rows flagged as an outlier on at least one column


In [20]:
outlier_records = food_banks_long_sd_records[food_banks_long_sd_records["Any_Outlier"]]

zip_summary_short = (
    outlier_records.groupby("Zip Code")
    .size()
    .rename("Times_Flagged")
    .reset_index()
)
zip_summary_short["Pct_of_Total_Outliers"] = (
    zip_summary_short["Times_Flagged"] / zip_summary_short["Times_Flagged"].sum() * 100
).round(1)
zip_summary_short = zip_summary_short.sort_values("Times_Flagged", ascending=False)

zip_summary_short

,Zip Code,Times_Flagged,Pct_of_Total_Outliers
4,91950,182,12.9
23,92105,159,11.3
0,91910,128,9.1
30,92121,103,7.3
5,91977,93,6.6
1,91911,86,6.1
35,92173,83,5.9
3,91945,81,5.8
14,92058,62,4.4
21,92102,61,4.3


In [21]:
outlier_by_source = (
    food_banks_long_sd_records.loc[food_banks_long_sd_records["Any_Outlier"], "Source"]
    .value_counts()
    .rename("Outlier_Count")
    .to_frame()
)

outlier_by_source["Pct_of_Total_Outliers"] = (
    outlier_by_source["Outlier_Count"] / outlier_by_source["Outlier_Count"].sum() * 100
).round(1)

print(outlier_by_source)

        Outlier_Count  Pct_of_Total_Outliers
Source                                      
SDFB              867                   61.7
FSD               539                   38.3


In [22]:
outlier_by_month = (
    food_banks_long_sd_records.loc[food_banks_long_sd_records["Any_Outlier"], "Month Year"]
    .value_counts()
    .rename("Outlier_Count")
    .to_frame()
)

outlier_by_month["Pct_of_Total_Outliers"] = (
    outlier_by_month["Outlier_Count"] / outlier_by_month["Outlier_Count"].sum() * 100
).round(1)

print(outlier_by_month)

            Outlier_Count  Pct_of_Total_Outliers
Month Year                                      
2020-09                26                    1.8
2020-12                26                    1.8
2021-03                26                    1.8
2021-04                26                    1.8
2020-06                24                    1.7
...                   ...                    ...
2019-02                 7                    0.5
2019-04                 7                    0.5
2019-01                 6                    0.4
2018-09                 6                    0.4
2018-11                 6                    0.4

[92 rows x 2 columns]


In [23]:
food_banks_long_sd_records.to_csv("FoodBank_cleaned.csv", index=False)